In [8]:
import pandas as pd
from tqdm import tqdm
import os, sys, json
tqdm.pandas()

## Txt to Parquet

In [9]:
def reformat_txt(root, file, json_file):
    with open(f"{root}/{json_file}") as f:
        columns = json.load(f)
    column_names = columns.keys()
    df = pd.read_csv(f"{root}/{file}", delimiter="|", index_col = False,
                     names=column_names, dtype=columns, encoding="latin")
    df.to_parquet(f"{root}/{file.replace("txt", "parquet")}", engine="pyarrow", index=False, compression="snappy")

# if __name__ == "__main__":
#     for root, dirs, files in os.walk("../data"):
#         for file in files:
#             if ".txt" in file:
#                 json_file = file.rsplit('_', 1)[0] + ".json"
#                 if json_file in files:
#                     print(f"processing {file}")
#                     try:
#                         reformat_txt(root, file, json_file)
#                     except Exception as e:
#                         print(e)

In [11]:
reformat_txt("../data/candidates", "candidate_master_2024.txt", "candidate_master.json")

## Tidy Corporate PACs

In [4]:
import re

In [98]:
def clean_corp(text, stopwords):
    text = re.sub(r"\(.*?\)", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = " ".join(word for word in text.split() if word.lower() not in stopwords)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [5]:
committee_dir = "../data/committees/"
years = range(2004, 2026, 2)
bus_stopwords = ["corp", "inc", "corporation", "llc", "plc", "ltd", "holdings", "co", "incorporated", "corporate"]
pac_stopwords = ["political action committee", "pac"]
stopwords = bus_stopwords + pac_stopwords

In [123]:
# Bi-annual corporate committees csv file
for year in years:
    committees = pd.read_parquet(f"../data/committees/committees_{year}.parquet")
    corporate = committees.loc[committees["ORG_TP"] == "C",]
    corporate = corporate.dropna(subset="CONNECTED_ORG_NM")
    corporate["CORPORATION"] = corporate["CONNECTED_ORG_NM"].apply(lambda x: clean_corp(x, stopwords))
    pacs = pd.read_parquet(f"../data/committees/pacs_{year}.parquet")
    list1 = corporate.columns.tolist()
    list2 = pacs.columns.tolist()
    match_cols = [col for col in list1 if col in list2]
    corporate = pd.merge(corporate, pacs, on = match_cols)
    corporate.to_csv(f"../data/committees/corporate_{year}.csv", index=False)

## Get Corporate Contributions

In [2]:
from datetime import datetime
def reformat_date(date, format="%m%d%Y"):
    try:
        date = datetime.strptime(date, format)
        return date.year, date.month, date.day
    except:
        return None, None, None

In [6]:
for year in years:
    corporate = pd.read_csv(f"../data/committees/corporate_{year}.csv")
    corp_id = corporate["CMTE_ID"].tolist()
    contrib_cand = pd.read_parquet(f"../data/contributions/committees/candidates_{year}.parquet")
    contrib = pd.read_parquet(f"../data/contributions/committees/committees_{year}.parquet")

    corp_cand = contrib_cand.loc[contrib_cand["CMTE_ID"].isin(corp_id),].reset_index(drop=True)
    corp_cand[["YEAR", "MONTH", "DAY"]] = corp_cand["TRANSACTION_DT"].apply(lambda x: pd.Series(reformat_date(x)))
    corp_cand.to_csv(f"../data/contributions/committees/corp_candidate_{year}.csv", index=False)

    corp_all = contrib.loc[contrib["CMTE_ID"].isin(corp_id),].reset_index(drop=True)
    corp_cand_id = corp_cand["SUB_ID"].tolist()

    corp_comm = corp_all.loc[~corp_all["SUB_ID"].isin(corp_cand_id),]
    corp_comm.to_csv(f"../data/contributions/committees/corp_comm_{year}.csv", index=False)

## Match Corporate PACs with Corporations

In [ ]:
committee_dir = "../data/committees/"
years = range(2004, 2026, 2)
cpacs = pd.DataFrame()
for year in years:
    corporate = pd.read_csv(f"{committee_dir}corporate_{year}.csv")
    corporate["YEAR"] = year
    cpacs = pd.concat([cpacs, corporate[["CMTE_ID", "CMTE_NM", "CONNECTED_ORG_NM", "CORPORATION", "YEAR"]]])

cpacs_full = cpacs.sort_values(by=["CMTE_ID", "YEAR"], ascending=True).reset_index(drop=True)
cpacs_short = cpacs_full.drop_duplicates(subset="CMTE_ID", keep="last")

cpacs_short.to_csv(f"{committee_dir}corporate_pacs.csv", index=False)
cpacs_full.to_csv(f"{committee_dir}/corporate_pacs_full.csv", index=False)

In [21]:
corporate = pd.read_csv(f"{committee_dir}corporate_2006.csv")
corporate["CVG_END_DT"]

0       12/31/2006
1       12/31/2006
2       10/31/2005
3       12/31/2006
4       12/31/2006
           ...    
1791    12/31/2006
1792           NaN
1793    12/31/2006
1794    12/31/2006
1795           NaN
Name: CVG_END_DT, Length: 1796, dtype: object

## Parse Large Individual Contribution Data

In [283]:
from pathlib import Path

contrib_dir = "../data/contributions/individual"
folder_path = Path(f"{contrib_dir}/individual_2022")
with open(f"{contrib_dir}/individual.json") as f:
    columns = json.load(f)
column_names = columns.keys()
for file in folder_path.iterdir():
    if "txt" in file.name and not os.path.exists(f"{folder_path}/{file.name.replace("txt","parquet")}"):
        file_dir = f"{folder_path}/{file.name}"
        try:
            individual = pd.read_csv(file_dir, delimiter="|", index_col = False,
                     names=column_names, dtype=columns, encoding="latin")
            individual.to_parquet(file_dir.replace("txt", "parquet"), engine="pyarrow", index=False, compression="snappy")
        except:
            continue

In [274]:
file = "../data/contributions/individual/individual_2020/itcont_2020_20200912_20200919.txt"
individual = pd.read_csv(file, delimiter="|", index_col = False,
                     names=column_names, dtype=columns, encoding="latin")


In [275]:
individual.to_parquet(file.replace("txt", "parquet"), engine="pyarrow", index=False, compression="snappy")

In [276]:
os.path.exists("../data/contributions/individual/individual_2020/test.txt")

True

In [279]:
contrib_dir = "../data/contributions/individual"
folder_path = Path(f"{contrib_dir}/individual_2020")
with open(f"{contrib_dir}/individual.json") as f:
    columns = json.load(f)
column_names = columns.keys()
for file in folder_path.iterdir():
    if "txt" in file:
        file_dir = f"{folder_path}/{file.name}"
        print(file_dir)

TypeError: argument of type 'WindowsPath' is not iterable